# ALLaM-7B-Instruct-preview on Kaggle T4×2 with `kaggle-vllm==0.2.0`

This notebook is a **first compatibility/inference run**, not a throughput benchmark.

It uses the public `kaggle-vllm` 0.2.0 workflow:

- Kaggle Notebook with **GPU T4 ×2**
- `kaggle-vllm[hub]==0.2.0`
- staged upstream vLLM runtime from the SDK's immutable binary profile
- **tensor parallelism = 2**
- **FP16** for Tesla T4 / SM75
- `enforce_eager=True`
- `disable_custom_all_reduce=True`
- expected SM75 attention path: **TRITON_ATTN**
- `humain-ai/ALLaM-7B-Instruct-preview`, pinned to the current `v2` revision and then resolved to an immutable Hugging Face commit

### Before running

1. Start a **fresh Kaggle GPU T4 ×2 session**.
2. Turn **Internet ON**.
3. Do **not** first load ALLaM with Transformers in the same Python process. The earlier Transformers experiment can retain most GPU memory and invalidate this vLLM run.
4. Run this notebook top-to-bottom.

The model is ~14 GB on the Hub, so the model-download cell will take time and disk space.

In [1]:
from pathlib import Path
import json
import os
import platform
import subprocess
import sys
import time

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
MODEL_TAG = "v2"

RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-allam-020")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

MODEL_CACHE = Path("/kaggle/working/hf-allam-cache")
RESULT_JSON = Path("/kaggle/working/allam-kaggle-vllm-tp2-result.json")
RUN_LOG = Path("/kaggle/working/allam-kaggle-vllm-tp2.log")
SCRIPT_PATH = Path("/kaggle/working/allam_kaggle_vllm_tp2.py")

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)

import torch
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
assert torch.cuda.is_available(), "CUDA is not available."
assert torch.cuda.device_count() == 2, (
    f"This notebook requires Kaggle GPU T4 x2; found {torch.cuda.device_count()} GPUs."
)
for i in range(2):
    p = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {p.name}, SM{p.major}{p.minor}, "
        f"{p.total_memory / 1024**3:.2f} GiB"
    )
    assert "T4" in p.name, f"GPU {i} is not a Tesla T4: {p.name}"
    assert (p.major, p.minor) == (7, 5), (
        f"GPU {i} is not SM75: {(p.major, p.minor)}"
    )

# A clean vLLM run should start with little GPU memory allocated.
subprocess.run(
    [
        "nvidia-smi",
        "--query-compute-apps=pid,process_name,used_memory",
        "--format=csv,noheader",
    ],
    check=False,
)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
GPU 0: Tesla T4 (UUID: GPU-7204c01e-4700-c3a6-8541-18ce2eb03b5b)
GPU 1: Tesla T4 (UUID: GPU-8cd47816-d6ea-9c77-dbb8-67cb4e80e421)
	GPU0	GPU1	CPU Affinity	NUMA Affinity	GPU NUMA ID
GPU0	 X 	PHB	0-3	0		N/A
GPU1	PHB	 X 	0-3	0		N/A

Legend:

  X    = Self
  SYS  = Connection traversing PCIe as well as the SMP interconnect between NUMA nodes (e.g., QPI/UPI)
  NODE = Connection traversing PCIe as well as the interconnect between PCIe Host Bridges within a NUMA node
  PHB  = Connection traversing PCIe as well as a PCIe Host Bridge (typically the CPU)
  PXB  = Connection traversing multiple PCIe bridges (without traversing the PCIe Host Bridge)
  PIX  = Connection traversing at most a single PCIe bridge
  NV#  = Connection traversing a bonded set of # NVLinks
Torch: 2.10.0+cu128 CUDA: 12.8
GPU 0: Tesla T4, SM75, 14.56 GiB
GPU 1: Tesla T4, SM75, 14.56 GiB


CompletedProcess(args=['nvidia-smi', '--query-compute-apps=pid,process_name,used_memory', '--format=csv,noheader'], returncode=0)

In [2]:
# ============================================================
# GPU HEALTH PREFLIGHT
#
# Run this FIRST in a fresh Kaggle T4 x2 session.
#
# Purpose:
# - detect ECC / CUDA device failures before expensive setup
# - verify each GPU independently
# - avoid wasting time downloading/loading a 7B model
# ============================================================

import os
import subprocess
import sys
import textwrap


print("=" * 72)
print("NVIDIA GPU INVENTORY")
print("=" * 72)

subprocess.run(
    ["nvidia-smi", "-L"],
    check=True,
)


print("\n" + "=" * 72)
print("NVIDIA ECC STATUS")
print("=" * 72)

ecc = subprocess.run(
    [
        "nvidia-smi",
        "-q",
        "-d",
        "ECC",
    ],
    capture_output=True,
    text=True,
)

print(ecc.stdout)

if ecc.stderr.strip():
    print("nvidia-smi stderr:")
    print(ecc.stderr)


# ------------------------------------------------------------
# Flag obvious uncorrectable ECC evidence
# ------------------------------------------------------------

ecc_lower = (
    ecc.stdout
    + "\n"
    + ecc.stderr
).lower()


danger_phrases = (
    "uncorrectable ecc error encountered",
    "double bit",
)


for phrase in danger_phrases:
    if phrase in ecc_lower:
        print(
            "\nWARNING: ECC output contains:",
            repr(phrase),
        )


# ============================================================
# Test each GPU in an isolated Python subprocess.
#
# This avoids creating CUDA contexts in the notebook process
# that later launches vLLM.
# ============================================================

GPU_TEST = r'''
import torch

assert torch.cuda.is_available()

props = torch.cuda.get_device_properties(0)

print(
    "GPU:",
    props.name,
    "SM",
    f"{props.major}{props.minor}",
)

# Force real GPU memory allocation + kernel execution.
x = torch.ones(
    (2048, 2048),
    device="cuda",
    dtype=torch.float16,
)

y = x @ x

torch.cuda.synchronize()

value = float(y[0, 0].item())

print("CUDA compute result:", value)
print("GPU CUDA HEALTH: PASS")
'''


failures = []


for physical_gpu in (0, 1):

    print(
        "\n" + "=" * 72
    )

    print(
        f"ISOLATED CUDA TEST — PHYSICAL GPU {physical_gpu}"
    )

    print(
        "=" * 72
    )

    env = os.environ.copy()

    # Present only one physical GPU to the child process.
    env[
        "CUDA_VISIBLE_DEVICES"
    ] = str(
        physical_gpu
    )


    probe = subprocess.run(
        [
            sys.executable,
            "-c",
            GPU_TEST,
        ],
        capture_output=True,
        text=True,
        env=env,
    )


    print(
        probe.stdout
    )

    if probe.stderr.strip():

        print(
            "stderr:"
        )

        print(
            probe.stderr
        )


    combined = (
        probe.stdout
        + "\n"
        + probe.stderr
    ).lower()


    bad = (
        probe.returncode != 0
        or "uncorrectable ecc"
        in combined
        or "cuda error"
        in combined
    )


    if bad:

        failures.append(
            {
                "gpu":
                    physical_gpu,

                "returncode":
                    probe.returncode,

                "output":
                    combined[-4000:],
            }
        )


# ============================================================
# Gate
# ============================================================

if failures:

    print(
        "\n" + "=" * 72
    )

    print(
        "GPU HEALTH PREFLIGHT: FAILED"
    )

    print(
        "=" * 72
    )

    for item in failures:

        print(
            f"\nGPU {item['gpu']} failed."
        )

        print(
            item["output"]
        )


    raise RuntimeError(
        "\nAt least one Kaggle GPU failed the isolated CUDA "
        "health test.\n\n"
        "DO NOT continue to kaggle-vllm bootstrap or Step 5.\n"
        "Stop this Kaggle session completely and request a "
        "fresh T4 x2 session."
    )


print(
    "\n" + "=" * 72
)

print(
    "GPU HEALTH PREFLIGHT: PASS"
)

print(
    "=" * 72
)

print(
    "Both T4 GPUs successfully completed isolated CUDA "
    "allocation and matrix-multiplication tests."
)

print(
    "It is safe to continue to kaggle-vllm bootstrap."
)

NVIDIA GPU INVENTORY
GPU 0: Tesla T4 (UUID: GPU-7204c01e-4700-c3a6-8541-18ce2eb03b5b)
GPU 1: Tesla T4 (UUID: GPU-8cd47816-d6ea-9c77-dbb8-67cb4e80e421)

NVIDIA ECC STATUS

==============NVSMI LOG==============

Timestamp                                              : Thu Oct  1 08:51:59 2026
Driver Version                                         : 580.159.04
CUDA Version                                           : 13.0

Attached GPUs                                          : 2
GPU 00000000:00:04.0
    ECC Mode
        Current                                        : Enabled
        Pending                                        : Enabled
    ECC Errors
        Volatile
            SRAM Correctable                           : 0
            SRAM Uncorrectable                         : 0
            DRAM Correctable                           : 0
            DRAM Uncorrectable                         : 0
        Aggregate
            SRAM Correctable                           : 0
         

## 1. Install exactly `kaggle-vllm==0.2.0`

The SDK is a small Python package. It does **not** replace Kaggle's PyTorch/CUDA stack with a normal `pip install vllm`. Its bootstrap step stages the project's verified native wheel and dependency overlay separately.

In [3]:
torch_before = subprocess.check_output(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda, torch.__file__)",
    ],
    text=True,
).strip()
print("Torch before SDK install:", torch_before)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-cache-dir",
        "kaggle-vllm[hub]==0.2.0",
    ],
    check=True,
)

import kaggle_vllm
print("kaggle-vllm:", kaggle_vllm.__version__)
assert kaggle_vllm.__version__ == "0.2.0"

Torch before SDK install: 2.10.0+cu128 12.8 /usr/local/lib/python3.12/dist-packages/torch/__init__.py
kaggle-vllm: 0.2.0


## 2. Strict bootstrap and native-runtime validation

A notebook-owned runtime path is used so this experiment does not overwrite other staged runtimes. The cell is rerunnable: if a completed manifest already exists, it reuses it instead of deleting anything.

In [4]:
# ============================================================
# STEP 2 — Bootstrap kaggle-vllm 0.2.0 and validate runtime
# ============================================================

from pathlib import Path
import json
import os
import subprocess
import sys

# These variables were defined in Step 0 / Step 1.
print("Runtime root :", RUNTIME_ROOT)
print("Staged path  :", STAGED)
print("Overlay path :", OVERLAY)
print("Manifest     :", MANIFEST)
print("Cache        :", CACHE)

# ------------------------------------------------------------
# 1. Bootstrap the kaggle-vllm runtime
# ------------------------------------------------------------

if MANIFEST.is_file():
    print("\nExisting completed runtime manifest found:")
    print(MANIFEST)
    print("Reusing the existing kaggle-vllm runtime.")

else:
    # Avoid silently overwriting a partial/unknown runtime.
    if STAGED.exists() or OVERLAY.exists():
        raise RuntimeError(
            "A partial kaggle-vllm runtime exists, but no completed manifest "
            "was found.\n"
            f"STAGED exists:  {STAGED.exists()}\n"
            f"OVERLAY exists: {OVERLAY.exists()}\n"
            f"MANIFEST exists:{MANIFEST.exists()}\n\n"
            "Start a fresh Kaggle session or use a different RUNTIME_ROOT."
        )

    RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    CACHE.mkdir(parents=True, exist_ok=True)

    bootstrap_cmd = [
        "kaggle-vllm",
        "bootstrap",
        "--strict",
        "--cache",
        str(CACHE),
        "--staged",
        str(STAGED),
        "--overlay",
        str(OVERLAY),
        "--manifest",
        str(MANIFEST),
    ]

    print("\n============================================================")
    print("BOOTSTRAP DRY RUN")
    print("============================================================")

    subprocess.run(
        bootstrap_cmd + ["--dry-run", "--json"],
        check=True,
    )

    print("\n============================================================")
    print("EXECUTING BOOTSTRAP")
    print("============================================================")

    subprocess.run(
        bootstrap_cmd,
        check=True,
    )


# ------------------------------------------------------------
# 2. Confirm bootstrap produced its manifest
# ------------------------------------------------------------

if not MANIFEST.is_file():
    raise RuntimeError(
        f"kaggle-vllm bootstrap finished without creating manifest:\n{MANIFEST}"
    )

print("\nBootstrap manifest exists:")
print(MANIFEST)


# ------------------------------------------------------------
# 3. Inspect manifest
# ------------------------------------------------------------

manifest_data = json.loads(
    MANIFEST.read_text(encoding="utf-8")
)

print("\n============================================================")
print("RUNTIME MANIFEST")
print("============================================================")
print(json.dumps(manifest_data, indent=2))


# ------------------------------------------------------------
# 4. Make this manifest the active kaggle-vllm runtime
# ------------------------------------------------------------

os.environ["KAGGLE_VLLM_MANIFEST"] = str(MANIFEST)

print("\nKAGGLE_VLLM_MANIFEST:")
print(os.environ["KAGGLE_VLLM_MANIFEST"])


# ------------------------------------------------------------
# 5. Activate the staged runtime in the notebook process
# ------------------------------------------------------------

from kaggle_vllm import activate_runtime

activated = activate_runtime(MANIFEST)

if not activated:
    raise RuntimeError(
        f"activate_runtime() failed for manifest:\n{MANIFEST}"
    )

print("\nRuntime activation: PASS")


# ------------------------------------------------------------
# 6. Verify vLLM Python + native CUDA modules
# ------------------------------------------------------------

import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

print("\n============================================================")
print("NATIVE vLLM IMPORT CHECK")
print("============================================================")

print(
    "vLLM distribution version:",
    getattr(vllm, "__version__", "unknown"),
)
print("vLLM path:", vllm.__file__)
print("vllm._C:", vllm._C.__file__)
print("vllm._moe_C:", vllm._moe_C.__file__)

print("\nNative vLLM imports: PASS")


# ------------------------------------------------------------
# 7. Verify Kaggle's original PyTorch remains intact
# ------------------------------------------------------------

torch_after = subprocess.check_output(
    [
        sys.executable,
        "-c",
        (
            "import os; "
            f"os.environ['KAGGLE_VLLM_MANIFEST']={str(MANIFEST)!r}; "
            "from kaggle_vllm import activate_runtime; "
            "assert activate_runtime("
            "os.environ['KAGGLE_VLLM_MANIFEST']"
            "); "
            "import torch; "
            "print("
            "torch.__version__, "
            "torch.version.cuda, "
            "torch.__file__"
            ")"
        ),
    ],
    text=True,
).strip()

print("\n============================================================")
print("PYTORCH PRESERVATION CHECK")
print("============================================================")

print("Torch before:", torch_before)
print("Torch after :", torch_after)

before_parts = torch_before.split()
after_parts = torch_after.split()

if before_parts[:2] != after_parts[:2]:
    raise RuntimeError(
        "PyTorch version/CUDA changed after kaggle-vllm bootstrap.\n"
        f"Before: {torch_before}\n"
        f"After : {torch_after}"
    )

print("\nPyTorch preservation: PASS")


# ------------------------------------------------------------
# 8. Run kaggle-vllm doctor
#
# IMPORTANT:
# kaggle-vllm 0.2.0 doctor DOES NOT accept --manifest.
# The active manifest is supplied through KAGGLE_VLLM_MANIFEST.
# ------------------------------------------------------------

doctor_json = Path(
    "/kaggle/working/kaggle-vllm-allam-doctor.json"
)

doctor_env = os.environ.copy()
doctor_env["KAGGLE_VLLM_MANIFEST"] = str(MANIFEST)

doctor_cmd = [
    "kaggle-vllm",
    "doctor",
    "--strict",
    "--json",
]

print("\n============================================================")
print("KAGGLE-vLLM STRICT DOCTOR")
print("============================================================")

with doctor_json.open(
    "w",
    encoding="utf-8",
) as f:
    proc = subprocess.run(
        doctor_cmd,
        stdout=f,
        stderr=subprocess.STDOUT,
        text=True,
        env=doctor_env,
    )

doctor_text = doctor_json.read_text(
    encoding="utf-8",
    errors="replace",
)

print(doctor_text)

if proc.returncode != 0:
    raise RuntimeError(
        "kaggle-vllm doctor failed.\n"
        f"Return code: {proc.returncode}\n"
        f"Log: {doctor_json}\n\n"
        f"{doctor_text}"
    )


# ------------------------------------------------------------
# 9. Final Step-2 status
# ------------------------------------------------------------

print("\n============================================================")
print("STEP 2: PASS")
print("============================================================")
print("kaggle-vllm 0.2.0 runtime is bootstrapped and activated.")
print("Native vLLM modules imported successfully.")
print("Kaggle PyTorch/CUDA stack was preserved.")
print("Strict doctor passed.")
print("Manifest:", MANIFEST)
print("Doctor log:", doctor_json)

Runtime root : /kaggle/working/kaggle-vllm-allam-020
Staged path  : /kaggle/working/kaggle-vllm-allam-020/vllm-staged
Overlay path : /kaggle/working/kaggle-vllm-allam-020/vllm-runtime-overlay
Manifest     : /kaggle/working/kaggle-vllm-allam-020/kaggle-vllm-runtime.json
Cache        : /kaggle/working/kaggle-vllm-cache

BOOTSTRAP DRY RUN
{
  "profile": "kaggle-t4x2-cu128",
  "strict": true,
  "compatible": true,
  "findings": [
    {
      "check": "Python implementation",
      "status": "pass",
      "message": "Python implementation: CPython"
    },
    {
      "check": "Python ABI",
      "status": "pass",
      "message": "Python ABI: cp312"
    },
    {
      "check": "operating system",
      "status": "pass",
      "message": "operating system: Linux"
    },
    {
      "check": "machine",
      "status": "pass",
      "message": "machine: x86_64"
    },
    {
      "check": "Kaggle runtime",
      "status": "pass",
      "message": "Kaggle runtime: True"
    },
    {
      "chec

Processing ./kaggle-vllm-cache/vllm-0.18.2.dev0+ga26e8dc7f.d20260822.cu128-cp312-cp312-linux_x86_64.whl
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 173.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.2/461.2 kB 326.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 347.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 177.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 323.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 808.1/808.1 kB 338.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 363.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.1/83.1 kB 261.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

## 3. Resolve ALLaM `v2` to an immutable Hub commit and download once

The Hugging Face model card identifies ALLaM as a 7B Llama-family Arabic/English model with a 4096-token model context and a BF16 training/config dtype. Tesla T4 (SM75) is not a native BF16 target, so the vLLM run below deliberately uses **FP16**, matching `kaggle-vllm`'s conservative T4 defaults.

The `v2` tag is first resolved to an immutable commit SHA; the actual model snapshot is then downloaded at that SHA. Only the model/tokenizer artifacts needed for inference are requested.

In [5]:
# ============================================================
# STEP 3A — Resolve ALLaM main -> immutable commit
#           and download the exact snapshot
# ============================================================

from huggingface_hub import HfApi, snapshot_download
from pathlib import Path
import json
import os

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"

# IMPORTANT:
# The model card documents revision="v2", but the current HF API
# does not resolve that tag for this repository.
#
# Therefore:
#   main -> current commit SHA -> immutable snapshot
#
# We then verify that config.json reports the v2 internal version.

hf_token = os.environ.get("HF_TOKEN") or None

api = HfApi(token=hf_token)

print("Resolving model repository:")
print(" ", MODEL_ID)

# Resolve current main branch to an immutable commit.
model_info = api.model_info(
    repo_id=MODEL_ID,
    revision="main",
)

MODEL_COMMIT = model_info.sha

if not MODEL_COMMIT:
    raise RuntimeError(
        f"Hugging Face returned no commit SHA for {MODEL_ID}"
    )

print("\nResolved:")
print("  branch : main")
print("  commit :", MODEL_COMMIT)

MODEL_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

print("\nDownloading immutable model snapshot...")

model_path = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_COMMIT,
        cache_dir=str(MODEL_CACHE),
        token=hf_token,

        # Deliberately download the complete repository snapshot.
        #
        # The weights dominate disk usage anyway, and avoiding a
        # restrictive allow_patterns list prevents accidental omission
        # of tokenizer/model metadata required by vLLM.
    )
).resolve()

if not model_path.is_dir():
    raise RuntimeError(
        f"snapshot_download did not produce a directory: {model_path}"
    )

print("\n============================================================")
print("ALLaM SNAPSHOT RESOLVED")
print("============================================================")

print("Model ID       :", MODEL_ID)
print("Immutable SHA  :", MODEL_COMMIT)
print("Local snapshot :", model_path)

print("\nTop-level files:")

for p in sorted(model_path.iterdir()):
    if p.is_file():
        size_gib = p.stat().st_size / (1024 ** 3)
        print(
            f"  {p.name:50s}"
            f" {size_gib:9.3f} GiB"
        )

print("\nSTEP 3A: PASS")

Resolving model repository:
  humain-ai/ALLaM-7B-Instruct-preview

Resolved:
  branch : main
  commit : a28dd1e67420cde72d3629c8633a974cf7d9c366



Fetching 466 files:   0%|          | 0/466 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

araMath_v3_5_shot.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.03G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]


ALLaM SNAPSHOT RESOLVED
Model ID       : humain-ai/ALLaM-7B-Instruct-preview
Immutable SHA  : a28dd1e67420cde72d3629c8633a974cf7d9c366
Local snapshot : /kaggle/working/hf-allam-cache/models--humain-ai--ALLaM-7B-Instruct-preview/snapshots/a28dd1e67420cde72d3629c8633a974cf7d9c366

Top-level files:
  .gitattributes                                         0.000 GiB
  README.md                                              0.000 GiB
  config.json                                            0.000 GiB
  generation_config.json                                 0.000 GiB
  model-00001-of-00003.safetensors                       4.635 GiB
  model-00002-of-00003.safetensors                       4.649 GiB
  model-00003-of-00003.safetensors                       3.756 GiB
  model.safetensors.index.json                           0.000 GiB
  tokenizer.model                                        0.001 GiB
  tokenizer_config.json                                  0.000 GiB

STEP 3A: PASS


In [6]:
# ============================================================
# STEP 3B — Validate downloaded ALLaM checkpoint
# ============================================================

CONFIG_PATH = model_path / "config.json"

if not CONFIG_PATH.is_file():
    raise RuntimeError(
        f"Missing config.json in model snapshot:\n{model_path}"
    )

config = json.loads(
    CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

print("============================================================")
print("ALLaM MODEL CONFIG")
print("============================================================")

print(json.dumps(
    config,
    indent=2,
    ensure_ascii=False,
))

# ------------------------------------------------------------
# Required architecture checks
# ------------------------------------------------------------

expected = {
    "model_type": "llama",
    "hidden_size": 4096,
    "num_hidden_layers": 32,
    "num_attention_heads": 32,
    "num_key_value_heads": 32,
    "max_position_embeddings": 4096,
}

for key, expected_value in expected.items():

    actual = config.get(key)

    if actual != expected_value:
        raise RuntimeError(
            f"Unexpected ALLaM config value:\n"
            f"  {key}\n"
            f"  expected: {expected_value!r}\n"
            f"  actual:   {actual!r}"
        )


architectures = config.get("architectures") or []

if "LlamaForCausalLM" not in architectures:
    raise RuntimeError(
        "Expected LlamaForCausalLM architecture, got:\n"
        f"{architectures}"
    )


# ------------------------------------------------------------
# Verify v2 model identity
# ------------------------------------------------------------

internal_version = config.get(
    "internal_version"
)

print("\nInternal model version:")
print(" ", internal_version)

if internal_version != "7b-alpha-v2.33.0.30":
    raise RuntimeError(
        "The downloaded model does not identify itself as the "
        "expected ALLaM v2 checkpoint.\n"
        f"internal_version={internal_version!r}"
    )


# ------------------------------------------------------------
# Runtime policy
# ------------------------------------------------------------

source_dtype = config.get(
    "torch_dtype"
)

print("\nModel source dtype :", source_dtype)
print("Runtime dtype      : float16")
print("Tensor parallelism : 2")
print("Initial context    : 2048")
print("Full model context :", config["max_position_embeddings"])

print(
    "\nNOTE: BF16 is recorded in the model config, "
    "but this T4/SM75 experiment deliberately uses FP16."
)

print("\nSTEP 3B: PASS")

ALLaM MODEL CONFIG
{
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 1,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.006,
  "intermediate_size": 11008,
  "max_position_embeddings": 4096,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": null,
  "rope_theta": 1000000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.39.3",
  "use_cache": true,
  "vocab_size": 64000,
  "internal_version": "7b-alpha-v2.33.0.30"
}

Internal model version:
  7b-alpha-v2.33.0.30

Model source dtype : bfloat16
Runtime dtype      : float16
Tensor parallelism : 2
Initial context    : 2048
Full model context : 4096

NOTE: BF16 is recorded in the model config, but this T4/SM75 experiment deliberately uses FP16.

STEP 3B: PASS


## 4. Write the standalone TP=2 inference script

The actual vLLM engine is launched in a **child process**. This mirrors the repository's acceptance/regression style and gives cleaner CUDA/vLLM worker teardown when the run finishes.

The script:

- activates the staged `kaggle-vllm` runtime;
- verifies two CUDA GPUs;
- creates `KaggleLLM(... tensor_parallel_size=2 ...)`;
- uses ALLaM's own chat template;
- runs one English and one Arabic request;
- records a compact JSON result;
- exits, allowing TP workers and GPU allocations to terminate with the child process.

In [7]:
# ============================================================
# STEP 4 — FIXED ALLaM TP=2 child-process runner
#
# Critical fix:
#   DO NOT initialize PyTorch CUDA before KaggleLLM/vLLM
#   creates its tensor-parallel workers.
# ============================================================

SCRIPT_SOURCE = r'''#!/usr/bin/env python3

from __future__ import annotations

import argparse
import json
import os
import platform
import subprocess
import sys
import time
from pathlib import Path


# ============================================================
# Environment MUST be configured before importing vLLM
# ============================================================

os.environ.setdefault(
    "TOKENIZERS_PARALLELISM",
    "false",
)

# vLLM normally defaults to fork for worker multiprocessing.
#
# This experiment explicitly uses spawn to avoid inheriting a
# pre-existing CUDA state into TP workers.
os.environ.setdefault(
    "VLLM_WORKER_MULTIPROC_METHOD",
    "spawn",
)

# If NCCL initialization fails again, preserve useful details.
os.environ.setdefault(
    "NCCL_DEBUG",
    "INFO",
)

# Helpful NCCL provenance.
os.environ.setdefault(
    "NCCL_DEBUG_SUBSYS",
    "INIT,ENV,GRAPH",
)


# ============================================================
# Activate kaggle-vllm runtime BEFORE importing vLLM
# ============================================================

from kaggle_vllm import (
    KaggleLLM,
    activate_runtime,
)

manifest = os.environ.get(
    "KAGGLE_VLLM_MANIFEST"
)

if not manifest:
    raise RuntimeError(
        "KAGGLE_VLLM_MANIFEST is not set."
    )

if not activate_runtime(manifest):
    raise RuntimeError(
        "Failed to activate kaggle-vllm runtime:\n"
        f"{manifest}"
    )


# Import upstream vLLM only after runtime activation.
import vllm

from vllm import SamplingParams


# ============================================================
# CLI
# ============================================================

def parse_args():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--model",
        required=True,
    )

    parser.add_argument(
        "--model-commit",
        required=True,
    )

    parser.add_argument(
        "--max-model-len",
        type=int,
        default=2048,
    )

    parser.add_argument(
        "--gpu-memory-utilization",
        type=float,
        default=0.85,
    )

    parser.add_argument(
        "--max-tokens",
        type=int,
        default=96,
    )

    parser.add_argument(
        "--result-json",
        required=True,
    )

    return parser.parse_args()


# ============================================================
# Main
# ============================================================

def main():

    args = parse_args()

    model_path = Path(
        args.model
    ).resolve()

    result_path = Path(
        args.result_json
    ).resolve()


    # --------------------------------------------------------
    # Model snapshot validation — CPU/filesystem only
    # --------------------------------------------------------

    if not model_path.is_dir():
        raise RuntimeError(
            f"Model directory does not exist:\n"
            f"{model_path}"
        )

    config_path = (
        model_path / "config.json"
    )

    if not config_path.is_file():
        raise RuntimeError(
            f"Missing config.json:\n"
            f"{config_path}"
        )

    config = json.loads(
        config_path.read_text(
            encoding="utf-8"
        )
    )

    if (
        config.get("model_type")
        != "llama"
    ):
        raise RuntimeError(
            "Unexpected model_type: "
            f"{config.get('model_type')!r}"
        )

    if (
        config.get("internal_version")
        != "7b-alpha-v2.33.0.30"
    ):
        raise RuntimeError(
            "Unexpected ALLaM internal_version: "
            f"{config.get('internal_version')!r}"
        )


    # --------------------------------------------------------
    # Hardware provenance WITHOUT torch.cuda initialization
    # --------------------------------------------------------

    gpu_list = subprocess.check_output(
        ["nvidia-smi", "-L"],
        text=True,
    ).strip()

    print(
        "============================================================",
        flush=True,
    )

    print(
        "PRE-vLLM ENVIRONMENT",
        flush=True,
    )

    print(
        "============================================================",
        flush=True,
    )

    print(
        "Python:",
        sys.version.replace("\n", " "),
        flush=True,
    )

    print(
        "Platform:",
        platform.platform(),
        flush=True,
    )

    print(
        "vLLM distribution:",
        getattr(
            vllm,
            "__version__",
            "unknown",
        ),
        flush=True,
    )

    print(
        "VLLM_WORKER_MULTIPROC_METHOD:",
        os.environ.get(
            "VLLM_WORKER_MULTIPROC_METHOD"
        ),
        flush=True,
    )

    print(
        "NCCL_DEBUG:",
        os.environ.get(
            "NCCL_DEBUG"
        ),
        flush=True,
    )

    print(
        "GPUs:",
        flush=True,
    )

    print(
        gpu_list,
        flush=True,
    )

    if gpu_list.count("Tesla T4") != 2:
        raise RuntimeError(
            "Expected exactly two Tesla T4 GPUs.\n"
            + gpu_list
        )


    # ========================================================
    # IMPORTANT
    #
    # Do NOT perform:
    #
    #   torch.cuda.is_available()
    #   torch.cuda.device_count()
    #   torch.cuda.get_device_properties()
    #   torch.cuda.get_device_name()
    #
    # before this point.
    #
    # Let vLLM own CUDA initialization for its TP workers.
    # ========================================================

    print(
        "\n============================================================",
        flush=True,
    )

    print(
        "INITIALIZING KaggleLLM TP=2",
        flush=True,
    )

    print(
        "============================================================",
        flush=True,
    )


    engine_started = (
        time.perf_counter()
    )

    llm = KaggleLLM(

        model=str(
            model_path
        ),

        # Actual two-GPU tensor parallelism.
        tensor_parallel_size=2,

        # Tesla T4 / SM75 compatibility.
        dtype="float16",

        # First compatibility run.
        max_model_len=
            args.max_model_len,

        gpu_memory_utilization=
            args.gpu_memory_utilization,

        # Validated kaggle-vllm T4 settings.
        enforce_eager=True,

        disable_custom_all_reduce=True,

        trust_remote_code=False,

        max_num_seqs=4,
    )


    engine_init_seconds = (
        time.perf_counter()
        - engine_started
    )


    print(
        "\nKaggleLLM TP=2 engine initialized.",
        flush=True,
    )

    print(
        "Engine initialization seconds:",
        round(
            engine_init_seconds,
            4,
        ),
        flush=True,
    )


    # --------------------------------------------------------
    # It is now safe to collect Torch/CUDA provenance because
    # the vLLM TP workers have already been established.
    # --------------------------------------------------------

    import torch

    torch_info = {
        "version":
            torch.__version__,

        "cuda":
            torch.version.cuda,

        "cuda_available":
            torch.cuda.is_available(),

        "visible_gpu_count":
            torch.cuda.device_count(),

        "gpus": [],
    }

    for index in range(
        torch.cuda.device_count()
    ):

        props = (
            torch.cuda.get_device_properties(
                index
            )
        )

        torch_info["gpus"].append(
            {
                "index":
                    index,

                "name":
                    props.name,

                "compute_capability":
                    f"{props.major}."
                    f"{props.minor}",

                "total_memory_gib":
                    round(
                        props.total_memory
                        / (1024 ** 3),
                        2,
                    ),
            }
        )


    print(
        "\nPost-engine CUDA provenance:",
        flush=True,
    )

    print(
        json.dumps(
            torch_info,
            indent=2,
        ),
        flush=True,
    )


    if (
        torch_info[
            "visible_gpu_count"
        ]
        != 2
    ):
        raise RuntimeError(
            "vLLM initialized, but the "
            "frontend does not see two GPUs."
        )


    # --------------------------------------------------------
    # Tokenizer
    # --------------------------------------------------------

    tokenizer = (
        llm.upstream.get_tokenizer()
    )

    print(
        "\nTokenizer:",
        tokenizer.__class__.__name__,
        flush=True,
    )


    # --------------------------------------------------------
    # English + Arabic prompts using checkpoint chat template
    # --------------------------------------------------------

    conversations = [

        [
            {
                "role": "user",
                "content":
                    "Who are you? "
                    "Answer in one concise sentence.",
            }
        ],

        [
            {
                "role": "user",
                "content":
                    "من أنت؟ "
                    "أجب بجملة واحدة مختصرة.",
            }
        ],
    ]


    prompts = []

    for messages in conversations:

        prompt = (
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

        if not prompt:
            raise RuntimeError(
                "ALLaM chat template returned "
                "an empty prompt."
            )

        prompts.append(
            prompt
        )


    for index, prompt in enumerate(
        prompts
    ):

        print(
            f"\nRendered prompt {index}:",
            repr(prompt),
            flush=True,
        )


    # --------------------------------------------------------
    # Deterministic generation
    # --------------------------------------------------------

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=args.max_tokens,
    )


    generation_started = (
        time.perf_counter()
    )

    outputs = llm.generate(
        prompts,
        sampling_params,
    )

    generation_seconds = (
        time.perf_counter()
        - generation_started
    )


    if len(outputs) != 2:
        raise RuntimeError(
            "Expected two vLLM outputs, got "
            f"{len(outputs)}."
        )


    records = []

    total_prompt_tokens = 0
    total_output_tokens = 0


    for index, request_output in enumerate(
        outputs
    ):

        if not request_output.outputs:
            raise RuntimeError(
                f"Request {index} returned "
                "no candidates."
            )

        candidate = (
            request_output.outputs[0]
        )

        generated_text = (
            candidate.text or ""
        ).strip()

        if not generated_text:
            raise RuntimeError(
                f"Request {index} returned "
                "empty generated text."
            )

        prompt_token_ids = (
            request_output.prompt_token_ids
            or []
        )

        output_token_ids = (
            candidate.token_ids
            or []
        )

        prompt_tokens = len(
            prompt_token_ids
        )

        output_tokens = len(
            output_token_ids
        )

        total_prompt_tokens += (
            prompt_tokens
        )

        total_output_tokens += (
            output_tokens
        )

        record = {
            "request_index":
                index,

            "prompt_tokens":
                prompt_tokens,

            "output_tokens":
                output_tokens,

            "finish_reason":
                getattr(
                    candidate,
                    "finish_reason",
                    None,
                ),

            "stop_reason":
                getattr(
                    candidate,
                    "stop_reason",
                    None,
                ),

            "text":
                generated_text,
        }

        records.append(
            record
        )

        print(
            f"\n=== REQUEST {index} ===",
            flush=True,
        )

        print(
            "Prompt tokens:",
            prompt_tokens,
            flush=True,
        )

        print(
            "Output tokens:",
            output_tokens,
            flush=True,
        )

        print(
            "Finish reason:",
            record["finish_reason"],
            flush=True,
        )

        print(
            "Response:",
            flush=True,
        )

        print(
            generated_text,
            flush=True,
        )


    # --------------------------------------------------------
    # Evidence
    # --------------------------------------------------------

    result = {

        "status":
            "PASS",

        "sdk":
            "kaggle-vllm==0.2.0",

        "model_repo":
            "humain-ai/"
            "ALLaM-7B-Instruct-preview",

        "model_path":
            str(model_path),

        "model_commit":
            args.model_commit,

        "internal_version":
            config.get(
                "internal_version"
            ),

        "tensor_parallel_size":
            2,

        "dtype":
            "float16",

        "max_model_len":
            args.max_model_len,

        "gpu_memory_utilization":
            args.gpu_memory_utilization,

        "enforce_eager":
            True,

        "disable_custom_all_reduce":
            True,

        "vllm_worker_multiproc_method":
            os.environ.get(
                "VLLM_WORKER_MULTIPROC_METHOD"
            ),

        "vllm_distribution_version":
            getattr(
                vllm,
                "__version__",
                "unknown",
            ),

        "torch":
            torch_info,

        "engine_init_seconds":
            round(
                engine_init_seconds,
                4,
            ),

        "generation_wall_seconds":
            round(
                generation_seconds,
                4,
            ),

        "total_prompt_tokens":
            total_prompt_tokens,

        "total_output_tokens":
            total_output_tokens,

        "outputs":
            records,
    }


    result_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temp_path = (
        result_path.with_suffix(
            result_path.suffix + ".tmp"
        )
    )

    temp_path.write_text(
        json.dumps(
            result,
            indent=2,
            ensure_ascii=False,
        )
        + "\n",
        encoding="utf-8",
    )

    temp_path.replace(
        result_path
    )


    print(
        "\n============================================================",
        flush=True,
    )

    print(
        "ALLaM TP=2 INFERENCE: PASS",
        flush=True,
    )

    print(
        "============================================================",
        flush=True,
    )

    print(
        "Result:",
        result_path,
        flush=True,
    )


if __name__ == "__main__":
    main()
'''


SCRIPT_PATH.write_text(
    SCRIPT_SOURCE,
    encoding="utf-8",
)


print(
    "Child runner written:"
)

print(
    SCRIPT_PATH
)


# ------------------------------------------------------------
# Syntax validation
# ------------------------------------------------------------

compile_result = subprocess.run(

    [
        sys.executable,
        "-m",
        "py_compile",
        str(SCRIPT_PATH),
    ],

    capture_output=True,

    text=True,
)


if compile_result.returncode != 0:

    print(
        compile_result.stdout
    )

    print(
        compile_result.stderr
    )

    raise RuntimeError(
        "Generated Step-4 runner "
        "failed syntax validation."
    )


print(
    "\nRunner syntax: PASS"
)

print(
    "STEP 4: PASS"
)

Child runner written:
/kaggle/working/allam_kaggle_vllm_tp2.py

Runner syntax: PASS
STEP 4: PASS


## 5. Execute ALLaM through `KaggleLLM` with real TP=2

The log should show both tensor-parallel workers/ranks and, on the validated T4 profile, vLLM selecting `TRITON_ATTN`.

This first run uses `max_model_len=2048` rather than immediately claiming full 4096-token compatibility. If this succeeds, 4096 can be tested as a separate capacity experiment.

In [8]:
# ============================================================
# STEP 5 — ALLaM TP=2 execution with hardware-fault detection
# ============================================================

import json
import os
import signal
import subprocess
import sys


# ------------------------------------------------------------
# Remove stale evidence only
# ------------------------------------------------------------

for path in (
    RESULT_JSON,
    RUN_LOG,
):
    if path.exists():
        path.unlink()


# ------------------------------------------------------------
# Child environment
# ------------------------------------------------------------

env = os.environ.copy()

env[
    "KAGGLE_VLLM_MANIFEST"
] = str(MANIFEST)

env[
    "TOKENIZERS_PARALLELISM"
] = "false"

env[
    "PYTHONUNBUFFERED"
] = "1"

env[
    "VLLM_WORKER_MULTIPROC_METHOD"
] = "spawn"

env[
    "NCCL_DEBUG"
] = "INFO"

env[
    "NCCL_DEBUG_SUBSYS"
] = "INIT,ENV,GRAPH"


# Remove accidental experimental NCCL overrides.
for key in (
    "NCCL_ALGO",
    "NCCL_PROTO",
):
    env.pop(
        key,
        None,
    )


# ------------------------------------------------------------
# Command
# ------------------------------------------------------------

cmd = [

    sys.executable,

    str(SCRIPT_PATH),

    "--model",
    str(model_path),

    "--model-commit",
    MODEL_COMMIT,

    "--max-model-len",
    "2048",

    "--gpu-memory-utilization",
    "0.85",

    "--max-tokens",
    "96",

    "--result-json",
    str(RESULT_JSON),
]


print(
    "=" * 72
)

print(
    "ALLaM-7B / kaggle-vllm 0.2.0 / REAL TP=2"
)

print(
    "=" * 72
)

print(
    "Model commit:",
    MODEL_COMMIT,
)

print(
    "Model:",
    model_path,
)

print(
    "Manifest:",
    MANIFEST,
)

print(
    "Multiprocessing:",
    env[
        "VLLM_WORKER_MULTIPROC_METHOD"
    ],
)


# ------------------------------------------------------------
# Run child process
# ------------------------------------------------------------

with RUN_LOG.open(
    "w",
    encoding="utf-8",
) as log:

    process = subprocess.Popen(

        cmd,

        stdout=log,

        stderr=subprocess.STDOUT,

        text=True,

        env=env,

        start_new_session=True,
    )


    try:

        return_code = process.wait(
            timeout=1200
        )


    except subprocess.TimeoutExpired:

        print(
            "Timeout: terminating vLLM process group."
        )

        try:

            os.killpg(
                process.pid,
                signal.SIGTERM,
            )

        except ProcessLookupError:
            pass


        try:

            process.wait(
                timeout=30
            )

        except subprocess.TimeoutExpired:

            try:

                os.killpg(
                    process.pid,
                    signal.SIGKILL,
                )

            except ProcessLookupError:
                pass

            process.wait()


        raise RuntimeError(
            "ALLaM TP=2 exceeded the "
            "1200-second timeout."
        )


# ------------------------------------------------------------
# Read complete log
# ------------------------------------------------------------

log_text = RUN_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)

log_lower = log_text.lower()


# ------------------------------------------------------------
# HARDWARE FAULT DETECTION
# ------------------------------------------------------------

ECC_SIGNATURES = (

    "uncorrectable ecc error encountered",

    "uncorrectable ecc",

    "double bit ecc",
)


detected_ecc = [

    signature

    for signature
    in ECC_SIGNATURES

    if signature
    in log_lower
]


if detected_ecc:

    print(
        "\n" + "=" * 72
    )

    print(
        "HARDWARE / GPU ECC FAILURE DETECTED"
    )

    print(
        "=" * 72
    )


    relevant = [

        line

        for line
        in log_text.splitlines()

        if (
            "ecc"
            in line.lower()

            or "nccl"
            in line.lower()

            or "cuda failure"
            in line.lower()
        )
    ]


    for line in relevant[-100:]:

        print(
            line
        )


    raise RuntimeError(
        "\nThe vLLM run encountered an UNCORRECTABLE "
        "GPU ECC error.\n\n"
        "This is not an ALLaM model/configuration error "
        "and should not be worked around by changing TP, "
        "dtype, max_model_len, Triton, or memory utilization.\n\n"
        "STOP this Kaggle GPU session and start a fresh "
        "T4 x2 session."
    )


# ------------------------------------------------------------
# Generic vLLM failure
# ------------------------------------------------------------

if return_code != 0:

    print(
        "\n" + "=" * 72
    )

    print(
        "vLLM ENGINE FAILED — NON-ECC ERROR"
    )

    print(
        "=" * 72
    )


    useful_terms = (

        "error",
        "exception",
        "runtimeerror",
        "nccl",
        "cuda",
        "triton",
        "oom",
        "out of memory",
        "attention",
    )


    useful = [

        line

        for line
        in log_text.splitlines()

        if any(
            term in line.lower()
            for term in useful_terms
        )
    ]


    for line in useful[-300:]:

        print(
            line
        )


    print(
        "\nFull log:",
        RUN_LOG,
    )


    raise RuntimeError(
        "ALLaM TP=2 failed for a non-ECC reason. "
        "Analyze RUN_LOG before changing parameters."
    )


# ------------------------------------------------------------
# Successful child must create evidence
# ------------------------------------------------------------

if not RESULT_JSON.is_file():

    raise RuntimeError(
        "vLLM exited successfully but RESULT_JSON "
        "was not created."
    )


result = json.loads(
    RESULT_JSON.read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Result validation
# ------------------------------------------------------------

assert (
    result.get("status")
    == "PASS"
)

assert (
    result.get(
        "tensor_parallel_size"
    )
    == 2
)

assert (
    result.get("dtype")
    == "float16"
)

assert (
    result.get(
        "model_commit"
    )
    == MODEL_COMMIT
)


outputs = result.get(
    "outputs",
    []
)

assert len(outputs) == 2

assert all(
    output.get(
        "text",
        "",
    ).strip()

    for output
    in outputs
)


print(
    "\n" + "=" * 72
)

print(
    "STEP 5: PASS"
)

print(
    "=" * 72
)

print(
    json.dumps(
        result,
        indent=2,
        ensure_ascii=False,
    )
)

ALLaM-7B / kaggle-vllm 0.2.0 / REAL TP=2
Model commit: a28dd1e67420cde72d3629c8633a974cf7d9c366
Model: /kaggle/working/hf-allam-cache/models--humain-ai--ALLaM-7B-Instruct-preview/snapshots/a28dd1e67420cde72d3629c8633a974cf7d9c366
Manifest: /kaggle/working/kaggle-vllm-allam-020/kaggle-vllm-runtime.json
Multiprocessing: spawn

STEP 5: PASS
{
  "status": "PASS",
  "sdk": "kaggle-vllm==0.2.0",
  "model_repo": "humain-ai/ALLaM-7B-Instruct-preview",
  "model_path": "/kaggle/working/hf-allam-cache/models--humain-ai--ALLaM-7B-Instruct-preview/snapshots/a28dd1e67420cde72d3629c8633a974cf7d9c366",
  "model_commit": "a28dd1e67420cde72d3629c8633a974cf7d9c366",
  "internal_version": "7b-alpha-v2.33.0.30",
  "tensor_parallel_size": 2,
  "dtype": "float16",
  "max_model_len": 2048,
  "gpu_memory_utilization": 0.85,
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "vllm_worker_multiproc_method": "spawn",
  "vllm_distribution_version": "0.18.2.dev0+ga26e8dc7f.d20260822",
  "torch": {
    

## 6. Check the evidence and worker cleanup

The run above proves **successful ALLaM generation under vLLM TP=2 only if the child process completed successfully**. It is not yet a performance claim.

Useful evidence to keep from the run:

- `/kaggle/working/allam-kaggle-vllm-tp2.log`
- `/kaggle/working/allam-kaggle-vllm-tp2-result.json`
- `/kaggle/working/kaggle-vllm-allam-doctor.json`
- the resolved immutable ALLaM commit printed earlier

The aggregate token/s value in the JSON is only a small smoke-test wall-clock measurement, **not** a controlled benchmark.

In [9]:
time.sleep(3)

print("Result JSON:")
print(RESULT_JSON.read_text(encoding="utf-8"))

print("\nProcesses that still mention the ALLaM runner/model path:")
ps = subprocess.run(
    ["ps", "-eo", "pid=,ppid=,args="],
    check=True,
    capture_output=True,
    text=True,
).stdout
matches = [
    line.strip()
    for line in ps.splitlines()
    if ("allam_kaggle_vllm_tp2.py" in line or str(model_path) in line)
    and str(os.getpid()) not in line
]
print("\n".join(matches) if matches else "None")

print("\nGPU state after child-process exit:")
subprocess.run(["nvidia-smi"], check=False)

Result JSON:
{
  "status": "PASS",
  "sdk": "kaggle-vllm==0.2.0",
  "model_repo": "humain-ai/ALLaM-7B-Instruct-preview",
  "model_path": "/kaggle/working/hf-allam-cache/models--humain-ai--ALLaM-7B-Instruct-preview/snapshots/a28dd1e67420cde72d3629c8633a974cf7d9c366",
  "model_commit": "a28dd1e67420cde72d3629c8633a974cf7d9c366",
  "internal_version": "7b-alpha-v2.33.0.30",
  "tensor_parallel_size": 2,
  "dtype": "float16",
  "max_model_len": 2048,
  "gpu_memory_utilization": 0.85,
  "enforce_eager": true,
  "disable_custom_all_reduce": true,
  "vllm_worker_multiproc_method": "spawn",
  "vllm_distribution_version": "0.18.2.dev0+ga26e8dc7f.d20260822",
  "torch": {
    "version": "2.10.0+cu128",
    "cuda": "12.8",
    "cuda_available": true,
    "visible_gpu_count": 2,
    "gpus": [
      {
        "index": 0,
        "name": "Tesla T4",
        "compute_capability": "7.5",
        "total_memory_gib": 14.56
      },
      {
        "index": 1,
        "name": "Tesla T4",
        "compute_c

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Interpretation

A successful notebook run supports the following narrow statement:

> `humain-ai/ALLaM-7B-Instruct-preview` at the recorded immutable revision loaded and generated English and Arabic text through `kaggle-vllm==0.2.0` using the SDK's staged upstream vLLM runtime, FP16, and `tensor_parallel_size=2` on the observed Kaggle dual-Tesla-T4/SM75 environment.

It does **not** by itself establish:

- TP=2 is faster than TP=1;
- the full 4096-token context is validated;
- concurrency scaling;
- TTFT / inter-token latency;
- production stability;
- output-quality superiority;
- a reusable vLLM `sharded_state` ALLaM artifact.

### If initialization fails

1. Confirm you started from a fresh Kaggle T4×2 session and did not leave a Transformers copy of ALLaM resident on the GPUs.
2. Inspect the log before changing settings.
3. If the error is KV-cache/context capacity related, retry with `--max-model-len 1024`.
4. Do **not** switch to BF16 on T4 simply because the source config says BF16.
5. Do **not** install a normal PyPI `vllm` over the Kaggle environment; this notebook is specifically testing the `kaggle-vllm` 0.2.0 runtime path.